# 05 — Label Generation and Feature Provenance Investigation

**Dataset:** `metaverse_transactions_dataset(1).csv`  
**Rows/columns expected:** 78,600 × 14  
**Target candidate:** `anomaly`  
**Reported origin:** Kaggle (exact Kaggle URL, dataset owner, version, license, and documentation still need to be recorded).

## Objective

Investigate whether the target label `anomaly` is independently supported by the observed fields, or whether it appears to have been assigned using `risk_score` or other feature rules.

This is a **forensic diagnostic notebook**, not a model-training notebook. It does not modify the original CSV, create new labels, or claim to recover the original author's code. Statistical association can identify suspicious patterns, but only the Kaggle dataset documentation, source code, or creator's explanation can establish the actual label-generation process.

## Important

- Keep the original CSV unchanged.
- Do not treat the Kaggle origin alone as proof that the data are real-world or independently labelled.
- Do not use `risk_score` as a predictive feature until its provenance and availability at prediction time are established.
- Complete the provenance decision register near the end of this notebook.

## 1. Load source and record provenance

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display

DATA_PATH = Path("metaverse_transactions_dataset.csv")
if not DATA_PATH.exists():
    csvs = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Upload the original CSV to this Colab session "
        f"or edit DATA_PATH. CSV files found: {[p.name for p in csvs]}"
    )

# Record the source file hash to identify the exact file version used.
sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df = pd.read_csv(DATA_PATH)

print("File:", DATA_PATH.name)
print("SHA-256:", sha256)
print("Shape:", df.shape)
display(df.head())

File: metaverse_transactions_dataset.csv
SHA-256: c7eae68df7ff627089b44df6123c4974e71065f4c342b45a4bf4664bdea6beb0
Shape: (78600, 14)


,timestamp,hour_of_day,sending_address,receiving_address,amount,transaction_type,location_region,ip_prefix,login_frequency,session_duration,purchase_pattern,age_group,risk_score,anomaly
0,2022-04-11 12:47:27,12,0x9d32d0bf2c00f41ce7ca01b66e174cc4dcb0c1da,0x39f82e1c09bc6d7baccc1e79e5621ff812f50572,796.949206,transfer,Europe,192.000,3,48,focused,established,18.75,low_risk
1,2022-06-14 19:12:46,19,0xd6e251c23cbf52dbd472f079147873e655d8096f,0x51e8fbe24f124e0e30a614e14401b9bbfed5384c,0.010000,purchase,South America,172.000,5,61,focused,established,25.00,low_risk
2,2022-01-18 16:26:59,16,0x2e0925b922fed01f6a85d213ae2718f54b8ca305,0x52c7911879f783d590af45bda0c0ef2b8536706f,778.197390,purchase,Asia,192.168,3,74,focused,established,31.25,low_risk
3,2022-06-15 09:20:04,9,0x93efefc25fcaf31d7695f28018d7a11ece55457f,0x8ac3b7bd531b3a833032f07d4e47c7af6ea7bace,300.838358,transfer,South America,172.000,8,111,high_value,veteran,36.75,low_risk
4,2022-02-18 14:35:30,14,0xad3b8de45d63f5cce28aef9a82cf30c397c6ceb9,0x6fdc047c2391615b3facd79b4588c7e9106e49f2,775.569344,sale,Africa,172.160,6,100,high_value,veteran,62.50,moderate_risk


### Kaggle source record — fill these fields from the Kaggle page

The user has identified Kaggle as the source, but the exact dataset page has not yet been supplied. Do not invent these details.

- Kaggle dataset URL: **TO FILL**
- Dataset title as shown on Kaggle: **TO FILL**
- Owner / uploader: **TO FILL**
- Version or update date: **TO FILL**
- License / usage terms: **TO FILL**
- Kaggle description and stated data-generation method: **TO FILL**
- Original source of transactions (if stated): **TO FILL**
- Does the page provide label definitions or generation code? **TO FILL**

Save a copy or screenshot of the Kaggle description, data dictionary, and any linked code/notebook. The Kaggle page's wording should be preserved rather than paraphrased as established fact.

## 2. Confirm schema, target, and basic integrity

In [ ]:
TARGET = "anomaly"
expected = [
    "timestamp", "hour_of_day", "sending_address", "receiving_address",
    "amount", "transaction_type", "location_region", "ip_prefix",
    "login_frequency", "session_duration", "purchase_pattern",
    "age_group", "risk_score", "anomaly"
]
print("Missing expected columns:", sorted(set(expected) - set(df.columns)))
print("Unexpected columns:", sorted(set(df.columns) - set(expected)))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Missing target labels:", int(df[TARGET].isna().sum()))
display(df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count"))
display((df[TARGET].value_counts(normalize=True, dropna=False)*100).round(3).rename("percent").to_frame())

Missing expected columns: []
Unexpected columns: []
Exact duplicate rows: 0
Missing target labels: 0


,count
anomaly,
low_risk,63494
moderate_risk,8611
high_risk,6495


,percent
anomaly,
low_risk,80.781
moderate_risk,10.955
high_risk,8.263


## 3. Test whether the target is a deterministic function of risk_score

In [ ]:
if "risk_score" in df.columns:
    print("Unique risk_score values:", df["risk_score"].nunique())
    display(df.groupby(TARGET)["risk_score"].agg(
        count="size", minimum="min", q01=lambda s: s.quantile(.01),
        median="median", q99=lambda s: s.quantile(.99), maximum="max",
        unique_values="nunique"
    ))
    # Check whether score ranges for classes overlap.
    score_ranges = df.groupby(TARGET)["risk_score"].agg(["min", "max"])
    display(score_ranges)
    classes = list(score_ranges.index)
    overlap_rows = []
    for i, a in enumerate(classes):
        for b in classes[i+1:]:
            lo = max(score_ranges.loc[a, "min"], score_ranges.loc[b, "min"])
            hi = min(score_ranges.loc[a, "max"], score_ranges.loc[b, "max"])
            overlap_rows.append({"class_a": a, "class_b": b,
                                 "range_overlap": bool(lo <= hi),
                                 "overlap_lower": lo if lo <= hi else np.nan,
                                 "overlap_upper": hi if lo <= hi else np.nan})
    display(pd.DataFrame(overlap_rows))
    print("Unique labels within each risk_score value:")
    score_label_counts = df.groupby("risk_score")[TARGET].nunique()
    print("Risk-score values associated with >1 target class:",
          int((score_label_counts > 1).sum()))
    print("Risk-score values associated with exactly 1 target class:",
          int((score_label_counts == 1).sum()))

Unique risk_score values: 31


,count,minimum,q01,median,q99,maximum,unique_values
anomaly,,,,,,,
high_risk,6495,90.0,90.0,100.0000,100.0000,100.0000,3
low_risk,63494,15.0,15.0,35.4375,59.0625,59.0625,24
moderate_risk,8611,62.5,62.5,67.5000,84.3750,84.3750,4


,min,max
anomaly,,
high_risk,90.0,100.0000
low_risk,15.0,59.0625
moderate_risk,62.5,84.3750


,class_a,class_b,range_overlap,overlap_lower,overlap_upper
0,high_risk,low_risk,False,NaN,NaN
1,high_risk,moderate_risk,False,NaN,NaN
2,low_risk,moderate_risk,False,NaN,NaN


Unique labels within each risk_score value:
Risk-score values associated with >1 target class: 0
Risk-score values associated with exactly 1 target class: 31


**Interpretation:** if each score value maps to exactly one target class and the class score ranges do not overlap, the target is perfectly recoverable from `risk_score` in this dataset. This is evidence of deterministic alignment, not proof of which variable was created first. The score might have generated the label, the label might have generated the score, or both might have been assigned from a third rule.

## 4. Examine categorical features against the target

In [ ]:
categorical_cols = [
    c for c in ["transaction_type", "location_region", "purchase_pattern", "age_group", "ip_prefix"]
    if c in df.columns
]
for col in categorical_cols:
    print(f"\\n### {col}")
    counts = pd.crosstab(df[col].fillna("<MISSING>").astype(str),
                         df[TARGET].fillna("<MISSING>").astype(str))
    row_pct = pd.crosstab(df[col].fillna("<MISSING>").astype(str),
                          df[TARGET].fillna("<MISSING>").astype(str),
                          normalize="index") * 100
    display(counts)
    display(row_pct.round(2))
    print("Distinct target classes per feature value:")
    display(df.groupby(col, dropna=False)[TARGET].nunique().rename("n_target_classes").to_frame())

\n### transaction_type


anomaly,high_risk,low_risk,moderate_risk
transaction_type,,,
phishing,2546,0,0
purchase,0,24940,0
sale,0,16429,8611
scam,3949,0,0
transfer,0,22125,0


anomaly,high_risk,low_risk,moderate_risk
transaction_type,,,
phishing,100.0,0.00,0.00
purchase,0.0,100.00,0.00
sale,0.0,65.61,34.39
scam,100.0,0.00,0.00
transfer,0.0,100.00,0.00


Distinct target classes per feature value:


,n_target_classes
transaction_type,
phishing,1
purchase,1
sale,2
scam,1
transfer,1


\n### location_region


... (15 outputs left out)

anomaly,high_risk,low_risk,moderate_risk
ip_prefix,,,
10.0,1237,12751,1727
172.0,1281,12685,1693
172.16,1329,12816,1766
192.0,1315,12568,1739
192.168,1333,12674,1686


anomaly,high_risk,low_risk,moderate_risk
ip_prefix,,,
10.0,7.87,81.14,10.99
172.0,8.18,81.01,10.81
172.16,8.35,80.55,11.10
192.0,8.42,80.45,11.13
192.168,8.49,80.76,10.74


Distinct target classes per feature value:


,n_target_classes
ip_prefix,
10.000,3
172.000,3
172.160,3
192.000,3
192.168,3


## 5. Investigate relationships among suspected co-generated fields

In [ ]:
pairs = [
    ("age_group", "purchase_pattern"),
    ("transaction_type", "anomaly"),
    ("purchase_pattern", "anomaly"),
    ("age_group", "anomaly"),
]
for a, b in pairs:
    if a in df.columns and b in df.columns:
        print(f"\\n### {a} versus {b}")
        display(pd.crosstab(df[a], df[b], margins=True))
        if a != TARGET and b != TARGET:
            print("Unique values of", b, "within each", a)
            display(df.groupby(a, dropna=False)[b].nunique().rename("n_unique").to_frame())

\n### age_group versus purchase_pattern


purchase_pattern,focused,high_value,random,All
age_group,,,,
established,26033,0,0,26033
new,0,0,26145,26145
veteran,0,26422,0,26422
All,26033,26422,26145,78600


Unique values of purchase_pattern within each age_group


,n_unique
age_group,
established,1
new,1
veteran,1


\n### transaction_type versus anomaly


anomaly,high_risk,low_risk,moderate_risk,All
transaction_type,,,,
phishing,2546,0,0,2546
purchase,0,24940,0,24940
sale,0,16429,8611,25040
scam,3949,0,0,3949
transfer,0,22125,0,22125
All,6495,63494,8611,78600


\n### purchase_pattern versus anomaly


anomaly,high_risk,low_risk,moderate_risk,All
purchase_pattern,,,,
focused,0,26033,0,26033
high_value,0,18205,8217,26422
random,6495,19256,394,26145
All,6495,63494,8611,78600


\n### age_group versus anomaly


anomaly,high_risk,low_risk,moderate_risk,All
age_group,,,,
established,0,26033,0,26033
new,6495,19256,394,26145
veteran,0,18205,8217,26422
All,6495,63494,8611,78600


## 6. Check numerical features and class-conditional patterns

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if c != TARGET]
summary_tables = {}
for col in numeric_cols:
    print(f"\\n### {col}")
    tab = df.groupby(TARGET)[col].agg(["count", "min", "median", "mean", "max", "std"])
    display(tab)
    summary_tables[col] = tab

\n### hour_of_day


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,0,12.0,11.641109,23,6.992830
low_risk,63494,0,12.0,12.134564,23,6.752026
moderate_risk,8611,0,5.0,7.012426,23,6.554707


\n### amount


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,0.01,491.667467,495.352890,1393.027468,243.012623
low_risk,63494,0.01,486.027151,486.214159,1557.150905,238.435717
moderate_risk,8611,0.01,701.855115,628.659679,1485.989339,265.112407


\n### ip_prefix


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,10.0,172.16,149.367603,192.168,68.200001
low_risk,63494,10.0,172.16,147.483633,192.168,69.496558
moderate_risk,8611,10.0,172.16,147.530346,192.168,69.466905


... (2 outputs left out)

\n### session_duration


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,20,30.0,29.462510,39,5.747063
low_risk,63494,20,59.0,67.574133,159,38.166378
moderate_risk,8611,20,118.0,115.584601,159,29.400959


\n### risk_score


,count,min,median,mean,max,std
anomaly,,,,,,
high_risk,6495,90.0,100.0000,97.686374,100.0000,3.422143
low_risk,63494,15.0,35.4375,36.274932,59.0625,11.834191
moderate_risk,8611,62.5,67.5000,69.200427,84.3750,6.774497


## 7. Check whether engineered-looking fields can be reconstructed

A field that can be reconstructed exactly from another column may be redundant. If it is also associated with the target, it may be a proxy for a label rule. This section tests only explicit, observable relationships; it does not infer hidden generation code.

In [ ]:
# Timestamp -> hour_of_day consistency
if "timestamp" in df.columns and "hour_of_day" in df.columns:
    parsed_ts = pd.to_datetime(df["timestamp"], errors="coerce")
    derived_hour = parsed_ts.dt.hour
    valid = parsed_ts.notna()
    print("Invalid timestamps:", int((~valid).sum()))
    print("hour_of_day mismatches:", int((df.loc[valid, "hour_of_day"].to_numpy() !=
                                           derived_hour.loc[valid].to_numpy()).sum()))

# Address self-transfers and repeated timestamps
if {"sending_address", "receiving_address"}.issubset(df.columns):
    self_transfer = df["sending_address"].astype(str).str.lower() == df["receiving_address"].astype(str).str.lower()
    print("Rows with same sender and receiver:", int(self_transfer.sum()))

if "timestamp" in df.columns:
    print("Repeated timestamp rows (beyond first occurrence):",
          int(df["timestamp"].duplicated(keep="first").sum()))

# Inspect ip_prefix values and type; values like 192.0 / 172.0 may be truncated prefixes,
# not complete IP addresses. Confirm from source documentation.
if "ip_prefix" in df.columns:
    print("ip_prefix dtype:", df["ip_prefix"].dtype)
    print("ip_prefix unique values:", sorted(df["ip_prefix"].dropna().unique().tolist()))
    print("ip_prefix value counts:")
    display(df["ip_prefix"].value_counts(dropna=False).to_frame("count"))

Invalid timestamps: 0
hour_of_day mismatches: 0
Rows with same sender and receiver: 87
Repeated timestamp rows (beyond first occurrence): 87
ip_prefix dtype: float64
ip_prefix unique values: [10.0, 172.0, 172.16, 192.0, 192.168]
ip_prefix value counts:


,count
ip_prefix,
172.160,15911
10.000,15715
192.168,15693
172.000,15659
192.000,15622


## 8. Check entity repetition and potential split contamination

In [ ]:
for col in ["sending_address", "receiving_address"]:
    if col in df.columns:
        vc = df[col].value_counts(dropna=False)
        print(f"\\n{col}: {df[col].nunique(dropna=False)} unique values")
        print("Values appearing more than once:", int((vc > 1).sum()))
        print("Rows belonging to repeated values:", int(vc[vc > 1].sum()))
        display(vc.head(10).rename("row_count").to_frame())

\nsending_address: 1161 unique values
Values appearing more than once: 1161
Rows belonging to repeated values: 78600


,row_count
sending_address,
0x4bfe6fa35149db081520aea5e503946572718756,232
0xb0b2f6fc707fbb7f9d27a9f4fe0cb0d6b39a0155,204
0xccbf5a9926606c77db8391185066a8ee3bc0a2af,203
0xf24dd92ee52bb0de1bab87766a4a3f1c17b2c4d9,203
0x91ee0c3b3f87d070e260f6208c52534d36817dae,203
0x1d132c165feddfa72ab5fca8b7dfa522efb53a56,203
0xceda6c769350a084f02a6e5f2fadf269ecc0c37d,203
0x09802f792bfbbf1f8515808f0fa68344c5625dbc,174
0x5b6124ac76b58955e1de4c58d87e5bca4c881aee,174


\nreceiving_address: 1166 unique values
Values appearing more than once: 1166
Rows belonging to repeated values: 78600


,row_count
receiving_address,
0x86b61ffe9ce4da504e6b7cf3f0925c1941265074,232
0x0496b829a3ede37a92aa2b9b101549ed23018cb8,203
0x897c07cac7f9149d0be00903c55d810f6e42c9cc,203
0x2c3cfb5537f82b3baf41b0869f20b66f47d57c23,203
0x617b83371b4c65debb6c2e90de89c827ddcb0884,203
0x90c61e65de8543a1e3daeda46d03fea897886666,203
0x4e3bf6cc6fa25ce2b65d184fc14d52f375768e3c,203
0xadc445c80282504e4af6afeb5681fded2c3d6069,203
0x880517c4e61f634eb26e59e702726e2cbae53959,203


**Interpretation:** repeated addresses can create entity overlap between train and test sets. This is not automatically leakage, but a random row split may overstate generalization to unseen wallets. For a future model, decide whether the intended deployment predicts future activity from known entities or generalizes to unseen entities. Use chronological and/or group-aware evaluation to match that deployment scenario.

## 9. Create an evidence-based feature risk register

In [ ]:
# This is a starting register, not an automatic final feature selection.
register = pd.DataFrame([
    ["risk_score", "Very high", "Perfect target alignment observed in earlier audit; score range separated classes", "Exclude from predictive inputs unless independent pre-label provenance is proven"],
    ["transaction_type", "High", "Strong target association observed; exact class-specific pattern requires review", "Hold out pending source/label rule verification"],
    ["purchase_pattern", "High", "Appears linked to age_group and target categories", "Hold out pending provenance verification"],
    ["age_group", "High", "Appears linked to purchase_pattern and target categories", "Hold out pending provenance verification"],
    ["login_frequency", "Medium / unknown", "Strong association observed in prior single-feature diagnostic", "Investigate generation process and prediction-time availability"],
    ["session_duration", "Medium / unknown", "Strong association observed in prior single-feature diagnostic", "Investigate generation process and prediction-time availability"],
    ["timestamp", "Unknown", "Temporal field; potential leakage if label reflects future outcomes or time-specific generation", "Use for chronology/splitting; derive permitted time features carefully"],
    ["hour_of_day", "Unknown", "Appears derivable from timestamp", "Check redundancy and prediction-time availability"],
    ["sending_address", "Identifier risk", "Repeated entity identifier", "Do not treat as ordinary numeric/categorical signal without justification; consider group split"],
    ["receiving_address", "Identifier risk", "Repeated entity identifier", "Do not treat as ordinary numeric/categorical signal without justification; consider group split"],
    ["amount", "Unknown", "Numeric transaction field", "Verify units, generation, and timing"],
    ["location_region", "Unknown", "Categorical context field", "Verify provenance and whether it is independently observed"],
    ["ip_prefix", "Unknown", "Stored as float with few unique values", "Verify representation and semantics from source documentation"],
], columns=["feature", "provisional_risk", "observed_reason", "provisional_action"])
display(register)
register.to_csv("05_feature_provenance_risk_register.csv", index=False)
print("Saved 05_feature_provenance_risk_register.csv")

,feature,provisional_risk,observed_reason,provisional_action
0,risk_score,Very high,Perfect target alignment observed in earlier a...,Exclude from predictive inputs unless independ...
1,transaction_type,High,Strong target association observed; exact clas...,Hold out pending source/label rule verification
2,purchase_pattern,High,Appears linked to age_group and target categories,Hold out pending provenance verification
3,age_group,High,Appears linked to purchase_pattern and target ...,Hold out pending provenance verification
4,login_frequency,Medium / unknown,Strong association observed in prior single-fe...,Investigate generation process and prediction-...
5,session_duration,Medium / unknown,Strong association observed in prior single-fe...,Investigate generation process and prediction-...
6,timestamp,Unknown,Temporal field; potential leakage if label ref...,Use for chronology/splitting; derive permitted...
7,hour_of_day,Unknown,Appears derivable from timestamp,Check redundancy and prediction-time availability
8,sending_address,Identifier risk,Repeated entity identifier,Do not treat as ordinary numeric/categorical s...
9,receiving_address,Identifier risk,Repeated entity identifier,Do not treat as ordinary numeric/categorical s...


Saved 05_feature_provenance_risk_register.csv


## 10. What this notebook can and cannot establish

### It can establish from the CSV
- Whether target classes align perfectly or near-perfectly with observed fields.
- Whether categorical fields have deterministic or near-deterministic mappings.
- Whether one field is reconstructible from another.
- Whether entity identifiers repeat and may affect evaluation design.
- Which features need source documentation before model use.

### It cannot establish from the CSV alone
- The original Kaggle creator's actual label-generation code.
- Whether the data are synthetic, simulated, anonymized, or collected from real transactions.
- Whether a field was available at the time a real prediction would have been made.
- Whether a strong association is causal or merely correlated.
- Whether the original labels represent independently verified fraud/anomaly outcomes.

To establish those points, obtain the exact Kaggle dataset URL, dataset description, data dictionary, linked notebooks/scripts, and any cited source paper.

## 11. Export a reproducibility summary

In [ ]:
from pathlib import Path
import json

summary = {
    "source_file": DATA_PATH.name,
    "sha256": sha256,
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target_column": TARGET,
    "target_counts": {str(k): int(v) for k, v in df[TARGET].value_counts(dropna=False).items()},
    "missing_values_total": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "kaggle_url": "TO FILL",
    "kaggle_owner": "TO FILL",
    "kaggle_version": "TO FILL",
    "license": "TO FILL",
}
Path("05_provenance_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved 05_provenance_summary.json")

{
  "source_file": "metaverse_transactions_dataset.csv",
  "sha256": "c7eae68df7ff627089b44df6123c4974e71065f4c342b45a4bf4664bdea6beb0",
  "rows": 78600,
  "columns": 14,
  "target_column": "anomaly",
  "target_counts": {
    "low_risk": 63494,
    "moderate_risk": 8611,
    "high_risk": 6495
  },
  "missing_values_total": 0,
  "exact_duplicate_rows": 0,
  "kaggle_url": "TO FILL",
  "kaggle_owner": "TO FILL",
  "kaggle_version": "TO FILL",
  "license": "TO FILL"
}
Saved 05_provenance_summary.json


## 12. Final decision gate

Do **not** proceed to a final ECIPS predictive-model claim until the following are completed:

- [ ] Exact Kaggle URL and uploader recorded.
- [ ] Kaggle description and license saved.
- [ ] Label definition for each `anomaly` class found or explicitly marked undocumented.
- [ ] `risk_score` generation rule found or explicitly marked unknown.
- [ ] Provenance of `transaction_type`, `purchase_pattern`, `age_group`, `login_frequency`, and `session_duration` checked.
- [ ] Prediction-time availability documented for every retained feature.
- [ ] Evaluation split selected to match intended deployment (future transactions, unseen entities, or both).
- [ ] Suspicious fields excluded or justified with documentary evidence.

**Research integrity note:** if the Kaggle page provides no label-generation documentation, report that limitation transparently. Do not claim that labels are ground-truth fraud labels unless the source supports that statement.